# Guarantees

This chapter is about what Kafi Streams guarantees you.

On the one hand, as for [Consistency](#consistency), Kafi Streams guarantees *strong consistency* instead of the *eventual consistency* guaranteed by classical stream processors like Kafka Streams and Flink.

But there is a but. In IT (let's say at least there), nothing comes for free.

Kafi Streams achieves strong consistency by using DBSP/pydbsp under the covers. For fault tolerance, Kafi Streams makes use of checkpointing the global pydbsp state. This, in turn, entails that Kafka transactions cannot really be utilized, and hence Kafi Streams does *not* guarantee *exactly-once delivery*. It only guarantees *at-least once delivery*. This is explained in [Delivery](#delivery).

In short, Kafi Streams guarantees, compared to classical stream processors:
* *strong consistency* instead of *eventual consistency*
* *at-least once delivery* instead of *exactly-once delivery*


## Overview

* [Consistency](#consistency)
  * [Internal consistency](#internal)
    * [Example](#example)
    * [Topology](#topology)
    * [Test](#test)
* [Delivery](#delivery)


---
<a id="consistency"></a>
## Consistency

Thanks to DBSP/pydbsp, Kafi Streams can guarantee *strong consistency* (https://algomaster.io/learn/system-design/strong-vs-eventual-consistency, https://jepsen.io/consistency/models).

This means essentially that the semantics of a Kafi Streams topology is the same as it would be in a standard database (if you map the operators of Kafi Streams to SQL).


<a id="internal"></a>
### Internal consistency

Strong consistency includes *internal consisteny* as defined by Jamie Brandon in his ingenious blog [Internal consistency in streaming systems](https://www.scattered-thoughts.net/writing/internal-consistency-in-streaming-systems/).

Jamie Brandon's blog showed how "classical" stream processors like Kafka Streams and Flink can easily be "tricked" by some simple lines of SQL and end up in emitting thousands of useless, and actually just *wrong* intermediate output messages.

At the time I read this blog for the first time (pretty late, early 2023), this was a revelation but I didn't yet know what to do about it. It seemed that everything that I had believed in about stream processing was off.


<a id="example"></a>
#### Example

How does the example work?

The input is a stream of transactions moving money between 10 distinct bank accounts. The amount of money moved is always 1.

First we create two views for the credits and debits of each account:

```sql
CREATE VIEW credits AS
SELECT
    to_account AS account, 
    sum(amount) AS credits
FROM
    transactions
GROUP BY
    to_account;

CREATE VIEW debits AS
SELECT
    from_account AS account, 
    sum(amount) AS debits
FROM
    transactions
GROUP BY
    from_account;
```    

Second, we calculate their balances (= credits - debits):

```sql
CREATE VIEW balance AS
SELECT
    credits.account AS account, 
    credits.credits - debits.debits AS balance
FROM
    credits
INNER JOIN debits 
    ON credits.account = debits.account;
```

Since money is only being moved around, never created or destroyed, the sum of all the balances should always be 0. This is how we get the total sum:

```sql
CREATE VIEW total AS
SELECT
    sum(balance)
FROM
    balance;
```

The blog is from 2021. Today, without using extra machinery or workarounds, classical stream processors like Flink still badly fail at this example!

There is, however, a novel stream processor by Hartmut Armbruster that *can* correctly handle it: [StoatFlow](https://stoatflow.io/) (see also Hartmut's [blog](https://stoatflow.io/blog/internal-consistency) about the topic).

 

<a id="topology"></a>
#### Topology

Here is a Kafi Streams topology modeling Jamie Brandon's example:

In [ ]:
import sys

sys.path.insert(1, "../..")

from kafi.streams.topologynode import TopologyNode as Tn

#

transaction_source_str = "transactions"
sink_str = "total"

transaction_tn = (
    Tn.source(transaction_source_str)
    .map(lambda r: {"from_account": r["from_account"],
                    "to_account": r["to_account"],
                    "amount": r["amount"]})
)
#
credits_tn = (
    transaction_tn
    .group_by_sum(
        key_fun=lambda r: r["to_account"],
        value_fun=lambda r: r["amount"],
        project_fun=lambda key_int, value_int: {"account": key_int,
                                                "credits": value_int})
)
#
debits_tn = (
    transaction_tn
    .group_by_sum(
        key_fun=lambda r: r["from_account"],
        value_fun=lambda r: r["amount"],
        project_fun=lambda key_int, value_int: {"account": key_int,
                                                "debits": value_int})
)
#
balance_tn = (
    credits_tn
    .join(debits_tn,
          left_key_fun=lambda l_r: l_r["account"],
          right_key_fun=lambda r_r: r_r["account"],
          project_fun=lambda l_r, r_r: {"account": l_r["account"],
                                             "balance": l_r["credits"] - r_r["debits"]})
)
#
sink_tn = (
    balance_tn.sum(value_fun=lambda x: x["balance"],
                   project_fun=lambda x: {"total": x})
).sink(sink_str)
#
_ = tn = Tn.build(sink_tn)


<a id="test"></a>
#### Test

Now let's throw some data at the topology to test it:

In [ ]:
import random

def gen(n_int):
    r_list = []
    for _ in range(n_int):
        r = {"from_account": random.randint(0, 9),
             "to_account": random.randint(0, 9),
             "amount": 1}
        #
        r_list.append(r)
    #
    return r_list

batch_int = 100
steps_int = 100

collected_m_list = []
for step_int in range(0, batch_int):
    m_list = tn.process({transaction_source_str: gen(100)}).get(sink_str, [])
    #
    print(f"\rStep {step_int + 1} ({(step_int + 1) * batch_int}/{batch_int * steps_int}): {m_list}")
    


As you can see, after the first `100` input records have been processed, Kafi Streams correctly returns:
```
{'total': 0}
```
And as this `total` doesn't change, however many further messages we throw at the topology, all the other outputs are empty.


---
<a id="delivery"></a>
## Delivery

Contrary to classical stream processors like Kafka Streams and Flink, Kafi Streams does not use individual state stores per operator but one global state, i.e., the state of the underlying pydbsp circuit.

Hence, saving a checkpoint to Kafka (or disk, S3 or Azure Blob Storage using Kafi's *emulated Kafka*) takes time.

Let's revisit how [checkpointing](checkpointing.ipynb) is implemented in the Kafi Streams consume + process + produce loop:

* before the loop: read the last checkpoint if there is any and set the global state and the offsets of the consumer group for the source topics accordingly,
* in the loop:
  1. consume new data from the source topics,
  2. process the new data and return the new resulting data,
  3. produce the new resulting data to the sink topics.
  4. if there is new resulting data and the checkpoint interval is exceeded:  
     4.1 save the checkpoint = the current global state + the offsets of the last processed messages from the source topics,  
     4.2 commit these offsets to Kafka.

Now consider how exactly-once semantics would have to be implemented with Kafka transactions:

* before the loop: read the last checkpoint if there is any and set the global state and the offsets of the consumer group for the source topics accordingly,
* in the loop:
  1. consume new data from the source topics,
  2. process the new data and return the new resulting data,
  3. ( start the Kafka transaction,  
     3.1. produce the new resulting data to the sink topics.  
     3.2. save the checkpoint = the current global state + the offsets of the last processed messages from the source topics,  
     3.3. commit the Kafka transaction )

What does this mean? To be able to use Kafka's transactions, we would have save the checkpoints in *every* step of our consume + process + consume loop. Infeasible.

Hence, to keep it simple, Kafi Streams only supports *at-least once* delivering guarantees.
